**Can a machine learning model predict a country's renewable energy share from its GDP and natural resource rents, and how well does it do?**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


df = pd.read_csv('merged_renewable_gdp_rents.csv') 

df["log_renew"]  = np.log(df["renewables_pct"].replace(0, np.nan)) #replaces 0 with NaN
df["log_gdp"]    = np.log(df["GDP_per_capita"])
df["log_natural_resource_rents"]    = np.log(df["natural_resource_rents_pct_gdp"].replace(0, np.nan)) 

features = ["log_gdp", "log_natural_resource_rents"]
target   = "log_renew"
 
data = df.dropna(subset=features + [target]) #Drops any row where any of the three columns is missing
X = data[features]
y = data[target]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42   #splits data into training (80%) and test (20%)
)
print(X_train.shape, X_test.shape)


In [17]:
lr = LinearRegression()
lr.fit(X_train, y_train)
 
y_pred = lr.predict(X_test)
 
print("R² on test:", r2_score(y_test, y_pred)) #R^2 gives variance in renewable share explained by the model
print("RMSE on test:", np.sqrt(mean_squared_error(y_test, y_pred))) #RMSE gives average size of the prediction error
# calculating RMSE manually because The squared=False argument for mean_squared_error was removed in newer versions of scikit-learn.
# Coefficients — same numbers you would see in OLS, different presentation
coef_df = pd.DataFrame({"feature": features, "coef": lr.coef_})
print(coef_df) #a coefficient of -0.05 on log_gdp means a 1% increase in GDP is associated with a 0.05% decrease in renewable share.


R² on test: 0.003345103185631193
RMSE on test: 1.90935264122221
                      feature      coef
0                     log_gdp -0.063461
1  log_natural_resource_rents -0.066793


In [18]:
from sklearn.tree import DecisionTreeRegressor
 
tree = DecisionTreeRegressor(max_depth=4, random_state=42) #limits depth of tree to 4
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
 
print("Tree R² on test:", r2_score(y_test, y_pred_tree))
print("Tree RMSE on test:", np.sqrt(mean_squared_error(y_test, y_pred_tree)))  
 
# Feature importance
imp = pd.DataFrame({"feature": features, "importance": tree.feature_importances_})
print(imp.sort_values("importance", ascending=False))

#Feature importance tells you how much each variable contributed to the tree's predictions, specifically, 
#how much each variable reduced prediction error across all the splits it was used in. A score of 1.0 means that variable 
#did all the work; 0.0 means it was never useful. 

Tree R² on test: 0.1355033796345042
Tree RMSE on test: 1.7782605492261698
                      feature  importance
1  log_natural_resource_rents    0.565632
0                     log_gdp    0.434368


A key limitation of the model is that we are trying to predict renewable share for each country using only GDP and resource rent data. A more nuanced model will incorporate more variables. 

**Honesty Checkist**

1. **Did I evaluate on data the model never saw?** Yes, the R^2 and RMSE values were calculated on the test data.
2. Yes, RMSE is reported for both models. However it's in units of log renewable share. A Tree RMSE of 0.17 on a log scale roughly means the model's predictions are off by about 17% in relative terms.
3. PArtially, this model compared linear regression against the decision tree, which is a valid comparison between a simple and a slightly more complex model. However a true baseline is missing, something like a model that simply predicts the average renewable share for every country regardless of inputs so we can show that this models actually does better than knowing none of the inputs.
4. Yes, in the markdown cell above. 
